In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import selfies as sf
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs, rdmolops, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import math
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
from sklearn.metrics import r2_score
from IPython.display import display
import seaborn as sns
import os
import re
torch.cuda.empty_cache()

import sys
sys.path.append('/home/andrze06/venvs/mlenv/lib/python3.10/site-packages/rdkit/Contrib/SA_Score')
import sascorer

In [2]:
df = pd.read_pickle('/data/home2/andrze06/projects/Smiles-latent-project/data/smiles_selfies_distance-matrix_dataset.pkl')

In [3]:
# ------------------------------------------------------------------
# BPE tokenizer over atomic SELFIES tokens (fast, Rust-backed)
# ------------------------------------------------------------------
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from tqdm.auto import tqdm


class BPETokenizer:
    """Byte-Pair-Encoding over atomic SELFIES tokens.

    Each unique SELFIES token is mapped to a unique Unicode char in the
    BMP private-use area, then HuggingFace's Rust BPE learns merges over
    those chars.  The mapping is bijective, so BPE ids are always fully
    reversible back to the original SELFIES tokens.
    """

    PAD_ID, SOS_ID, EOS_ID = 0, 1, 2

    def __init__(self):
        self.tok2char = {}      # SELFIES token -> single unicode char
        self.char2tok = {}      # reverse
        self.hf_tokenizer = None
        self.vocab_size = 3     # PAD, SOS, EOS

    # --------------------------------------------------------------
    def train(self, selfies_list, num_merges=100, min_frequency=2):
        # 1. collect atomic SELFIES tokens
        unique_tokens = set()
        for s in tqdm(selfies_list, desc="scanning SELFIES tokens"):
            unique_tokens.update(sf.split_selfies(s))
        unique_tokens = sorted(unique_tokens)

        if len(unique_tokens) >= 6400:
            raise ValueError(
                f"Too many unique SELFIES tokens ({len(unique_tokens)}); "
                "widen the char mapping range."
            )

        # 2. build bijective char mapping
        for i, t in enumerate(tqdm(unique_tokens, desc="building char map")):
            c = chr(0xE000 + i)
            self.tok2char[t] = c
            self.char2tok[c] = t

        # 3. corpus: each molecule is one "word" of private-use chars
        corpus = [
            "".join(self.tok2char[t] for t in sf.split_selfies(s))
            for s in tqdm(selfies_list, desc="tokenizing corpus")
        ]

        # 4. train Rust BPE
        self.hf_tokenizer = Tokenizer(models.BPE())
        self.hf_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

        trainer = trainers.BpeTrainer(
            vocab_size=len(unique_tokens) + num_merges,
            min_frequency=min_frequency,
            initial_alphabet=sorted(self.tok2char.values()),
            show_progress=True,
        )
        self.hf_tokenizer.train_from_iterator(
            corpus,
            trainer=trainer,
            length=len(corpus),          # <- makes the progress bar real
        )

        # HF ids 0..V-1, we reserve 0..2 for PAD/SOS/EOS
        self.vocab_size = self.hf_tokenizer.get_vocab_size() + 3

    # --------------------------------------------------------------
    def encode(self, selfie_tokens):
        """list of SELFIES tokens -> list of BPE ids (no specials)."""
        text = "".join(self.tok2char[t] for t in selfie_tokens)
        ids = self.hf_tokenizer.encode(text).ids
        return [i + 3 for i in ids]

    # --------------------------------------------------------------
    def decode_to_selfie_tokens(self, ids):
        """list of BPE ids -> list of original SELFIES tokens."""
        shifted = [int(i) - 3 for i in ids if int(i) >= 3]
        pieces = self.hf_tokenizer.convert_ids_to_tokens(shifted)
        out = []
        for p in pieces:
            for c in p:                     # skip any space markers
                tok = self.char2tok.get(c)
                if tok is not None:
                    out.append(tok)
        return out


# ---------------- Data & Loaders -------------------------------------
df['tokens'] = df['selfies'].apply(lambda x: list(sf.split_selfies(x)))

tokenizer = BPETokenizer()
tokenizer.train(df['selfies'].tolist(), num_merges=200)

vocab_size = tokenizer.vocab_size
PAD, SOS, EOS = BPETokenizer.PAD_ID, BPETokenizer.SOS_ID, BPETokenizer.EOS_ID


def molecule_tok2id(tokens, tokenizer):
    return np.array([SOS] + tokenizer.encode(tokens) + [EOS])


df['token_ids'] = [
    molecule_tok2id(toks, tokenizer)
    for toks in tqdm(df['tokens'].tolist(), desc="encoding dataset")
]
df['lengths'] = df['tokens'].apply(lambda x: len(x))

# padding
sequences = df['token_ids'].tolist()
max_len = max(len(seq) for seq in sequences)

X_train, X_temp, D_train, D_temp = train_test_split(
    sequences, df['distance_matrix'].tolist(),
    test_size=0.2, random_state=42, shuffle=True,
)
X_val, X_test, D_val, D_test = train_test_split(
    X_temp, D_temp, test_size=0.5, random_state=42, shuffle=True,
)


class MoleculeDataset(Dataset):
    def __init__(self, X, D):
        self.X = X
        self.D = D

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        tokens = self.X[index]
        dist_matrix = self.D[index]
        return (torch.tensor(tokens, dtype=torch.long),
                torch.tensor(dist_matrix, dtype=torch.long))


train_data = MoleculeDataset(X_train, D_train)
val_data = MoleculeDataset(X_val, D_val)
test_data = MoleculeDataset(X_test, D_test)
data = MoleculeDataset(sequences, df['distance_matrix'].tolist())

print(f"BPE vocab size: {vocab_size}  |  max sequence length: {max_len}")

scanning SELFIES tokens:   0%|          | 0/794403 [00:00<?, ?it/s]

building char map:   0%|          | 0/139 [00:00<?, ?it/s]

tokenizing corpus:   0%|          | 0/794403 [00:00<?, ?it/s]

encoding dataset:   0%|          | 0/794403 [00:00<?, ?it/s]

BPE vocab size: 342  |  max sequence length: 45


In [4]:
# utils
def collate_fn(batch):
    tokens_list, dist_list = zip(*batch)

    max_seq  = max(len(t)          for t          in tokens_list)
    max_dist = max(d.shape[0]      for d          in dist_list)

    # tokens are integer ids -> long ; distances are float
    padded_sequence = torch.zeros(len(batch), max_seq, dtype=torch.long)
    padded_dist     = torch.zeros(len(batch), max_dist + 1, max_dist + 1, dtype=torch.float32)

    for i, (tokens, dist) in enumerate(zip(tokens_list, dist_list)):
        n = dist.shape[0]
        padded_sequence[i, :len(tokens)] = tokens
        padded_dist[i, 1:n+1, 1:n+1] = dist

    return padded_sequence, padded_dist


@torch.no_grad()
def accuracy(model, loader, mode='eval', pad_id=0, device='cuda'):
    model.eval()

    total_correct = 0
    total_tok = 0
    total_seq = 0
    perfect = 0

    for x, D in loader:
        x, D = x.to(device), D.to(device)

        if mode == 'train':
            logits, _, _, _ = model(x, D, mode=mode)
            targets = x[:, 1:]

            pred = logits.argmax(dim=-1)

            mask = (targets != pad_id)

            correct = (pred == targets) & mask

            total_correct += correct.sum().item()
            total_tok += mask.sum().item()

            seq_correct = (correct.sum(dim=1) == mask.sum(dim=1))
            perfect += seq_correct.sum().item()
            total_seq += x.size(0)

        else:
            tokens, _, _, _ = model(x, D, mode='eval')

            for i, pred in enumerate(tokens):
                true = x[i]
                mask = (true != pad_id)
                true_len = mask.sum().item()

                if true_len == 0:
                    continue
                if len(pred) < true_len:
                    pad = torch.full((true_len - len(pred),), pad_id, device=pred.device, dtype=pred.dtype)
                    pred = torch.cat([pred, pad], dim=0)
                else:
                    pred = pred[:true_len]

                true = true[:true_len]
                correct = (pred == true)
                total_correct += correct.sum().item()
                total_tok += true_len
                perfect += int(correct.all())
                total_seq += 1

    return (
        total_correct / max(total_tok, 1),
        perfect / max(total_seq, 1)
    )

def set_token_ticks(ax, tokens, max_show=30):
    n = len(tokens)

    step = max(1, n // max_show)
    idxs = list(range(0, n, step))

    labels = [tokens[i] for i in idxs]

    ax.set_xticks(idxs)
    ax.set_yticks(idxs)

    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)

def cleaned_selfie(tokens, pad="<PAD>", sos="<SOS>", eos="<EOS>"):
    out = []
    for t in tokens:
        if t == eos:
            break
        if t in (pad, sos):
            continue
        out.append(t)
    return out

In [5]:
# model
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        pe = torch.zeros(max_len, hidden_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_size, 2).float() * (-math.log(10000.0) / hidden_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        if x.dim() == 3:
            B, T, _ = x.shape
        elif x.dim() == 2:
            B, T = x.shape
        if T <= self.pe.size(0):
            pe = self.pe[:T]  
        else:
            device = x.device
            H = self.hidden_size
            position = torch.arange(T, dtype=torch.float, device=device).unsqueeze(1)  
            div_term = torch.exp(torch.arange(0, H, 2, device=device).float() * (-math.log(10000.0)/H))
            pe = torch.zeros(T, H, device=device)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.d = hidden_size // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_v = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_o = nn.Linear(hidden_size, hidden_size, bias=False)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, 2*hidden_size),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(2*hidden_size, hidden_size),
            nn.Dropout(p=0.1)
        )
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, q, k, v, pad_mask=None):   # [B, T, H]
        B, T_q, H = q.shape
        _, T_v, _ = v.shape
        Q = self.W_q(q)
        K = self.W_k(k)
        V = self.W_v(v)
        Q = Q.view(B, self.num_heads, T_q, self.d)
        K = K.view(B, self.num_heads, T_v, self.d)
        V = V.view(B, self.num_heads, T_v, self.d)

        attn_logits = torch.einsum('baih,bajh->baij', Q, K)

        if pad_mask is not None:
            key_mask = pad_mask[:, None, None, :]
            attn_logits = attn_logits.masked_fill(~key_mask, float('-inf'))

        attn = F.softmax(attn_logits / math.sqrt(self.d), dim=-1)

        if pad_mask is not None:
            query_mask = pad_mask[:, None, :, None]
            attn = attn * query_mask.float()

        h = torch.einsum('baij,bajh->baih',attn, V)
        h = h.view(B, T_q, H)
        h = self.W_o(h)
        
        h = q + self.W_o(h)
        h = h + self.ff(self.norm2(h))
        h = self.norm1(h)
        return h, attn

class MultiSlotPooling(nn.Module):
    def __init__(self, hidden_size, num_slots):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(num_slots, hidden_size))
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_v = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, h, mask):
        k = self.W_k(h)
        v = self.W_v(h)
        attn = torch.einsum("kd,btd->bkt", self.queries, k)
        attn = attn.masked_fill(~mask[:, None, :], -1e9)
        attn = F.softmax(attn, dim=-1)
        slots = torch.einsum("bkt,btd->bkd", attn, v)
        return slots


class VaeTransformer(nn.Module):
    def __init__(self, vocab_size, hidden_size, latent_size, max_len, attn_heads=8, num_slots=8, encoder_layers=1, decoder_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_slots = num_slots
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.pos_encoder = PositionalEmbedding(max_len, hidden_size)
        self.conv_pos = nn.Conv1d(hidden_size, hidden_size, kernel_size=3, padding=1, groups=hidden_size)
        
        # Encoder
        self.pos_block = MultiHeadAttention(hidden_size, attn_heads)
        self.encoder_blocks = nn.ModuleList([MultiHeadAttention(hidden_size, attn_heads) for _ in range(encoder_layers)])
        self.pool = MultiSlotPooling(hidden_size, num_slots=num_slots)
        self.slots_mix = MultiHeadAttention(hidden_size, num_heads=num_slots)

        self.slot_gamma = nn.Parameter(torch.ones(1, num_slots, hidden_size))
        self.slot_beta = nn.Parameter(torch.zeros(1, num_slots, hidden_size))

        # VAE heads
        self.slot_mu = nn.Linear(hidden_size, latent_size)
        self.slot_logvar = nn.Linear(hidden_size, latent_size)  
        
        self.slot_compress_mu = nn.Linear(hidden_size, latent_size // num_slots)
        self.slot_compress_logvar = nn.Linear(hidden_size, latent_size // num_slots)

        self.fc_mu = nn.Linear(hidden_size, latent_size)
        self.fc_logvar = nn.Linear(hidden_size, latent_size)
        
        # pooling in latent space with uncertainty
        self.latent_query = nn.Parameter(torch.randn(1, 1, latent_size))
        self.latent_key = nn.Linear(latent_size, latent_size)

        # Decoder
        self.max_len = max_len
        self.z_to_slot = nn.Linear(hidden_size // num_slots, hidden_size)
        self.decoder_embed = nn.Embedding(vocab_size, hidden_size)
        self.decoder_pos = PositionalEmbedding(max_len, hidden_size)

        self.z_to_memory = nn.Linear(latent_size, hidden_size)
        self.slots_to_memory = nn.Linear(latent_size // num_slots, hidden_size)

        self.decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=attn_heads,
            dim_feedforward=2 * hidden_size,
            batch_first=True
        )

        self.decoder_transformer = nn.TransformerDecoder(
            self.decoder_layer,
            num_layers=decoder_layers
        )
        # Output head
        self.fc_output = nn.Linear(hidden_size, vocab_size)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def causal_mask(self, T, device):
        return torch.triu(
            torch.ones(T, T, device=device),
            diagonal=1
        ).bool()
    
    def encode(self, x, D, mode=None):  
        B, _ = x.shape
        h = self.embedding(x)
        fourier_pos_encoding = self.pos_encoder(x)
        h = h + fourier_pos_encoding

        mask = (x != 0)
        
        for block in self.encoder_blocks:
            h, attn_matrix = block(h, h, h, pad_mask=mask)
            
        h = h.masked_fill(~mask[:, :, None], 0.0)

        slots = self.pool(h, mask)
        slots = F.layer_norm(slots, slots.shape[-1:])

        B, K, Z = slots.shape

        slots_mu = self.slot_mu(slots)
        slots_logvar = self.slot_logvar(slots)

        # Latent pool from slots with uncertainty
        q = F.normalize(self.latent_query.expand(B, 1, Z), dim=-1)
        k = F.normalize(self.latent_key(slots_mu), dim=-1)

        logits = torch.einsum("bqz,bkz->bqk", q, k)

        confidence = -torch.logsumexp(slots_logvar, dim=-1).unsqueeze(1)
        confidence_scale = 0.5

        logits = logits + confidence_scale * confidence
        attn = torch.softmax(logits / 0.5, dim=-1)

        mu = torch.einsum("bqk,bkz->bqz", attn, slots_mu).squeeze(1)

        var = torch.exp(slots_logvar)
        var_agg = torch.einsum("bqk,bkz->bqz", attn, var).squeeze(1)
        logvar = torch.log(var_agg + 1e-8)
        
        if mode == "test":
            return mu, logvar, attn_matrix
        else:
            return mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def decode(self, z, x_in=None, max_len=80, start_id=1, eos_id=2):
        B = z.size(0)
        device = z.device

        memory = self.z_to_memory(z).unsqueeze(1)

        if x_in is not None:
            x_emb = self.decoder_embed(x_in)
            x_emb = x_emb + self.decoder_pos(x_emb)

            T = x_emb.size(1)
            tgt_mask = self.causal_mask(T, device)

            h = self.decoder_transformer(
                tgt=x_emb,
                memory=memory,
                tgt_mask=tgt_mask
            )

            logits = self.fc_output(h)   
            return logits

        else:
            tokens = torch.full((B, 1), start_id, dtype=torch.long, device=device)
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            max_len = self.max_len * 2

            for _ in range(max_len):
                x_emb = self.decoder_embed(tokens)
                x_emb = x_emb + self.decoder_pos(x_emb)

                T = tokens.size(1)
                tgt_mask = self.causal_mask(T, device)

                h = self.decoder_transformer(
                    tgt=x_emb,
                    memory=memory,
                    tgt_mask=tgt_mask
                )

                logits_step = self.fc_output(h[:, -1])

                next_token = torch.argmax(logits_step, dim=-1, keepdim=True)

                next_token = torch.where(
                    finished.unsqueeze(1),
                    torch.full_like(next_token, eos_id),
                    next_token
                )

                tokens = torch.cat([tokens, next_token], dim=1)

                finished |= (next_token.squeeze(1) == eos_id)

                if finished.all():
                    break

            return tokens
            
    def forward(self, x, D, mode='eval'):
        mu, logvar = self.encode(x, D)
        z = self.reparameterize(mu, logvar)

        if mode == 'train':
            x_in = x[:, :-1]

            logits = self.decode(z, x_in=x_in)

            return logits, mu, logvar, z
    
        if mode == "eval":
            tokens = self.decode(z, x_in=None)

            return tokens, mu, logvar, z
        
        if mode == "test":
            mu, logvar, attn_matrix = self.encode(x, D, mode="test")
            z = self.reparameterize(mu, logvar)

            tokens = self.decode(z, x_in=None)

            return tokens, mu, logvar, attn_matrix, z


def vae_loss(logits, targets, mu, logvar, beta=0.01, pad_id=0):
    B, T, V = logits.shape

    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)

    rec_loss = F.cross_entropy(
        logits,
        targets,
        ignore_index=pad_id
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return rec_loss + beta * kl, rec_loss, kl

In [6]:
# training
device = 'cuda' if torch.cuda.is_available() else 'cpu'

hidden_size = 256
attention_heads = 8
num_slots = 8
encoder_layers = 3
decoder_layers = 2
latent_size = 256

model = VaeTransformer(vocab_size, hidden_size, latent_size, max_len, attn_heads=attention_heads, num_slots=num_slots, encoder_layers=encoder_layers, decoder_layers=decoder_layers).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

beta = 0.01
epochs = 50
batch_size = 256
history = []

train_loader = DataLoader(train_data, batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_data, batch_size, shuffle=False, collate_fn=collate_fn)

for epoch in range(1, epochs+1):
    model.train()
    total = 0
    total_rec = 0
    total_kl = 0
    total_val = 0
    pbar = tqdm(train_loader)

    for x, D in pbar:
        x, D = x.to(device), D.to(device) 
        logits, mu, logvar, z = model(x, D=None, mode='train')
        x = x[:, 1:]
        loss, rec, kl = vae_loss(logits, x, mu, logvar, beta=beta)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        pbar.set_postfix({
            "rec": f"{rec.item():.3f}",
            "kl": f"{kl.item():.3f}", 
            "tot": f"{loss.item():.3f}",
        })
        total += loss.item()
        total_rec += rec.item()
        total_kl += kl.item()

    with torch.no_grad():
        model.eval()
        for x, D in val_loader:
            x, D = x.to(device), D.to(device)
            logits, mu, logvar, z = model(x, D, mode='train')
            x = x[:, 1:]
            val_loss, val_rec, val_kl = vae_loss(
                logits, x, mu, logvar, 
                beta=beta
            )
            total_val += val_loss.item()

    token_acc, seq_acc = accuracy(model, val_loader, mode='train')
    total = total / len(train_loader)
    total_rec = total_rec / len(train_loader)
    total_kl = total_kl / len(train_loader)
    total_val = total_val / len(val_loader)
    history.append((total, total_rec, total_kl, total_val))
    print(f"Epoch: {epoch:03d} | total={total:.4f} | rec={total_rec:.4f} | kl={total_kl:.4f} | val={total_val:.4f} | token_acc={token_acc*100:.2f}% | seq_acc={seq_acc*100:.2f}%")

  0%|          | 0/2483 [00:00<?, ?it/s]

Epoch: 001 | total=1.8785 | rec=1.8549 | kl=2.3608 | val=0.4609 | token_acc=86.90% | seq_acc=37.23%


  0%|          | 0/2483 [00:00<?, ?it/s]

Epoch: 002 | total=0.5006 | rec=0.4729 | kl=2.7763 | val=0.2183 | token_acc=94.06% | seq_acc=58.90%


  0%|          | 0/2483 [00:00<?, ?it/s]

...

Epoch: 048 | total=0.0448 | rec=0.0307 | kl=1.4073 | val=0.0228 | token_acc=99.72% | seq_acc=96.60%


  0%|          | 0/2483 [00:00<?, ?it/s]

Epoch: 049 | total=0.0444 | rec=0.0304 | kl=1.3985 | val=0.0233 | token_acc=99.70% | seq_acc=96.41%


  0%|          | 0/2483 [00:00<?, ?it/s]

Epoch: 050 | total=0.0439 | rec=0.0300 | kl=1.3900 | val=0.0225 | token_acc=99.72% | seq_acc=96.57%


In [ ]:
# save the model
torch.save({
    "model_state": model.state_dict(),
    "bpe_merges": tokenizer.merges,
    "bpe_expansion": tokenizer.expansion,
    "selfie_tok2id": tokenizer.selfie_tok2id,
}, "/data/home2/andrze06/projects/Smiles-latent-project/ProbeVAE/trained_models/specified-selfies-bpe.pt")

AttributeError: 'BPETokenizer' object has no attribute 'expansion'

In [7]:
# utils
def get_molecule_properties(mol):
    return {
        "molWt": Descriptors.MolWt(mol),
        "HeavyAtomCount": mol.GetNumHeavyAtoms(),
        "cLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "NumRotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRingCount": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "NumSpiroAtoms": rdMolDescriptors.CalcNumSpiroAtoms(mol),
        "NumBridgeheadAtoms": rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
        "BertzCT": Descriptors.BertzCT(mol),
        "QED": QED.qed(mol),
        "SA-score": sascorer.calculateScore(mol)
    }

def compute_property(mol, name):
    if name == "molWt":
        return Descriptors.MolWt(mol)
    elif name == "HeavyAtomCount":
        return mol.GetNumHeavyAtoms()
    elif name == "cLogP":
        return Descriptors.MolLogP(mol)
    elif name == "TPSA":
        return Descriptors.TPSA(mol)
    elif name == "HBD":
        return rdMolDescriptors.CalcNumHBD(mol)
    elif name == "HBA":
        return rdMolDescriptors.CalcNumHBA(mol)
    elif name == "NumRotatableBonds":
        return rdMolDescriptors.CalcNumRotatableBonds(mol)
    elif name == "RingCount":
        return rdMolDescriptors.CalcNumRings(mol)
    elif name == "AromaticRingCount":
        return rdMolDescriptors.CalcNumAromaticRings(mol)
    elif name == "FractionCSP3":
        return rdMolDescriptors.CalcFractionCSP3(mol)
    elif name == "NumSpiroAtoms":
        return rdMolDescriptors.CalcNumSpiroAtoms(mol)
    elif name == "NumBridgeheadAtoms":
        return rdMolDescriptors.CalcNumBridgeheadAtoms(mol)
    elif name == "BertzCT":
        return Descriptors.BertzCT(mol)
    elif name == "QED":
        return QED.qed(mol)
    elif name == "SA-score":
        return sascorer.calculateScore(mol)
    else:
        return np.nan

def clean_selfie_ids(ids):
    """Strip SOS / EOS / PAD from a generated id sequence."""
    result = []
    for tok in ids:
        if tok == SOS:
            continue
        if tok == EOS or tok == PAD:
            return result
        result.append(tok)
    return result

def latents_to_mol(model, z, tokenizer):
    """Decode latents -> BPE ids -> SELFIES tokens -> SELFIES -> RDKit mol."""
    ids = model.decode(z)
    mols = []
    for seq in ids.cpu().numpy():
        bpe_ids = clean_selfie_ids(seq)
        selfie_tokens = tokenizer.decode_to_selfie_tokens(bpe_ids)
        selfie_str = ''.join(selfie_tokens)
        try:
            smi = sf.decoder(selfie_str)
            mol = Chem.MolFromSmiles(smi)
            mols.append(mol)
        except Exception:
            mols.append(None)
    return mols

def set_token_ticks(ax, tokens, max_show=30):
    n = len(tokens)

    step = max(1, n // max_show)
    idxs = list(range(0, n, step))

    labels = [tokens[i] for i in idxs]

    ax.set_xticks(idxs)
    ax.set_yticks(idxs)

    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)

def cleaned_selfie(tokens, pad="<PAD>", sos="<SOS>", eos="<EOS>"):
    out = []
    for t in tokens:
        if t == eos:
            break
        if t in (pad, sos):
            continue
        out.append(t)
    return out

In [8]:
Y = pd.read_csv('/data/home2/andrze06/projects/Smiles-latent-project/data/molecule_properties.csv')
X = pd.read_csv('/data/home2/andrze06/projects/Smiles-latent-project/data/confound_features.csv')    

In [9]:
# extract latents for linear probe
def extract_latents(data, model):
    loader = DataLoader(data, batch_size=128, shuffle=False, collate_fn=collate_fn)
    zs = []
    model.eval()
    with torch.no_grad():
        for x, D in tqdm(loader):
            x, D = x.to(device), D.to(device)
            mu, _ = model.encode(x, D)
            zs.append(mu.cpu().numpy())
    z = np.concatenate(zs, axis=0)
    return z

z = extract_latents(data, model)

  0%|          | 0/6207 [00:00<?, ?it/s]

In [10]:
directions = []
for target_name in Y.columns:
    print(f"\n=== Predicting {target_name} ===")
    y = Y[target_name].values
    z_train, z_test, y_train, y_test = train_test_split(z, y, test_size=0.2, random_state=42)

    linear_probe = Pipeline([
        ("scaler", StandardScaler()),
        ("linear", LinearRegression())
    ])

    linear_probe.fit(z_train, y_train)
    y_pred = linear_probe.predict(z_test)
    print("R2:", r2_score(y_test, y_pred))
    w = linear_probe.named_steps['linear'].coef_ / linear_probe.named_steps['scaler'].scale_
    w /= np.linalg.norm(w)
    directions.append(w)


=== Predicting molWt ===
R2: 0.9118647685308503

=== Predicting HeavyAtomCount ===
R2: 0.9519668817520142

=== Predicting cLogP ===
R2: 0.5867421411551637

=== Predicting TPSA ===
R2: 0.633548367539579

=== Predicting HBD ===
R2: 0.5968016386032104

=== Predicting HBA ===
R2: 0.6629606485366821

=== Predicting NumRotatableBonds ===
R2: 0.7560611963272095

=== Predicting RingCount ===
R2: 0.8891697525978088

=== Predicting AromaticRingCount ===
R2: 0.7824572324752808

=== Predicting FractionCSP3 ===
R2: 0.6733333226781321

=== Predicting NumSpiroAtoms ===
R2: 0.12380421161651611

=== Predicting NumBridgeheadAtoms ===
R2: 0.1868559718132019

=== Predicting BertzCT ===
R2: 0.9047138083835369

=== Predicting QED ===
R2: 0.36768261868343133

=== Predicting SA-score ===
R2: 0.6564760221641301
